# Full pipeline: score 25 episodes, threshold, push to Hub

This walks the full curation flow against `lerobot/aloha_mobile_cabinet`
(85 episodes; we'll score the first 25). The notebook is split so you
can stop after the score histogram if you don't want to actually push
anything to the Hub.

Total runtime: ~10 minutes against a fast hosted API. ~$0.30 in API spend.

## Step 1 — Environment + imports

In [ ]:
import os, sys
sys.path.insert(0, '..')

for var in ('CRUCIBLE_VLM_ENDPOINT', 'CRUCIBLE_VLM_MODEL'):
    print(f'  {var:30s} = {os.environ.get(var, "<not set>")}')

assert os.environ.get('CRUCIBLE_VLM_ENDPOINT'), 'set CRUCIBLE_VLM_ENDPOINT first'

In [ ]:
import asyncio, json
import pandas as pd
import plotly.express as px
from src.config import CrucibleConfig
from src.pipeline import score_dataset
from src.filtering import select_episodes, push_filtered_to_hub

## Step 2 — Score 25 episodes

If you've run this dataset before, the cache at
`data/precached/lerobot__aloha_mobile_cabinet.json` will short-circuit
the LLM calls. Pass `use_cache=False` to force a fresh score.

In [ ]:
REPO = 'lerobot/aloha_mobile_cabinet'

cfg = CrucibleConfig()
cfg.max_episodes_per_run = 25
cfg.frames_per_episode = 12

def progress(i, total, ep):
    v = ep.get('verdict') or {}
    print(f"  [{i:3d}/{total}] ep {ep['episode_index']:3d}  score={v.get('final_score'):.2f}  {v.get('verdict')}")

results = asyncio.run(score_dataset(REPO, cfg, progress_callback=progress, use_cache=True))
print(f'\nTotal: {len(results)} episodes scored')

## Step 3 — Tabulate + plot the verdict distribution

In [ ]:
df = pd.DataFrame([
    {
        'episode': r['episode_index'],
        'duration_s': r['duration_s'],
        'score': float(r['verdict']['final_score']),
        'verdict': r['verdict']['verdict'],
        'top_concern': r['verdict'].get('top_concern') or '',
    }
    for r in results
])
df.head(10)

In [ ]:
fig = px.histogram(
    df,
    x='score',
    color='verdict',
    nbins=20,
    title=f'Crucible verdict distribution: {REPO} ({len(df)} episodes)',
    color_discrete_map={'KEEP': '#16a34a', 'POLISH': '#eab308', 'REJECT': '#dc2626'},
)
fig.show()

## Step 4 — Apply a threshold and inspect

`select_episodes` enforces both `score >= threshold` and
`verdict != REJECT`. Drag the threshold to the value that gives you
the kept-rate you want.

In [ ]:
THRESHOLD = 7.0

kept, dropped = select_episodes(results, threshold=THRESHOLD)
print(f'Threshold: {THRESHOLD}')
print(f'  Kept:    {len(kept)} ({len(kept) / max(len(results), 1):.0%})')
print(f'  Dropped: {len(dropped)}')
print()
print('Top 5 dropped episodes (worst first):')
dropped_sorted = sorted(dropped, key=lambda r: r['verdict']['final_score'])
for r in dropped_sorted[:5]:
    v = r['verdict']
    print(f"  ep {r['episode_index']:3d}  score={v['final_score']:.2f}  {v['verdict']:6s}  {v.get('top_concern') or v['summary'][:80]}")

## Step 5 — (Optional) Push the curated subset to your own Hub repo

**Skip this step if you don't want to push anything.** It writes a new
HuggingFace dataset under your account.

Requires:
- A write-scoped HF token (https://huggingface.co/settings/tokens)
- A target repo name in the form `username/dataset_name`
- The source repo's chunked parquet/video shards mirrored into your
  new repo (Crucible handles this; expect a few minutes for upload)

The pushed repo carries a `crucible_curation` block in `info.json` so
downstream consumers can load only the kept episodes via
`LeRobotDataset(repo, episodes=kept_indices)`.

In [ ]:
HF_TOKEN = os.environ.get('HF_TOKEN', '')
TARGET_REPO = ''  # e.g. 'your-username/aloha_mobile_cabinet-curated'

if not HF_TOKEN or not TARGET_REPO:
    print('Skipping push: set HF_TOKEN env var and TARGET_REPO above to enable.')
else:
    push_result = asyncio.run(push_filtered_to_hub(
        source_repo=REPO,
        results=results,
        threshold=THRESHOLD,
        target_repo=TARGET_REPO,
        hf_token=HF_TOKEN,
    ))
    print(json.dumps(push_result, indent=2, default=str))

## Verifying the curated subset loads back

If you pushed in step 5, the new repo can be loaded via the official
`lerobot` library:

```python
from lerobot import LeRobotDataset
import json
from huggingface_hub import hf_hub_download

info_path = hf_hub_download(TARGET_REPO, 'meta/info.json', repo_type='dataset')
kept = json.load(open(info_path))['crucible_curation']['kept_episode_indices']
ds = LeRobotDataset(TARGET_REPO, episodes=kept)
print(f'frames in curated subset: {len(ds)}')
```